# Create Images of Enemy Bullet Sprites

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:60% !important; }</style>"))

Load in the sprite data

In [1]:
import re
from colors_map import *

sprites_files = ["uridium/src/explosion_sprites.asm"]
sprites_data = {}
sprites_bytes = {}
sprites_ram = {}
sprites_offset = {}
sprite_orientation = {}
sprite_ram = 0x5000
for sprites_file in sprites_files:
    input_file = open(sprites_file,'r')
    sprite_data = []
    sprite_bytes = ""
    for l in input_file.readlines():
        if "SPRITE" in l:
            if sprite_data:
                sprites_data[sprite_name] = sprite_data
                sprites_bytes[sprite_name] = sprite_bytes
                sprites_ram[sprite_name] = hex(sprite_ram)[2:].upper()
                sprite_ram += 0x40
            sprite_name = l[22:36].strip()
            sprite_orientation[sprite_name] = (l[42:45],l[51:54],l[60:63],)
            sprites_offset[sprite_name] = l[18:20]
            sprite_data = []
            sprite_bytes = ""
            continue

        m = re.findall(r"[0-1]{24}",l)
        if not m:
            continue
        bits = m[0]
        sprite_line = []
        for i in range(0,23,2):
            bitpair = bits[i:i+2]
            sprite_line += [bitpair]
            sprite_line += [bitpair]
        sprite_data += [sprite_line]
        sprite_bytes += l[10:21]+'\n'
    if sprite_data:
        sprites_data[sprite_name] = sprite_data
        sprites_bytes[sprite_name] = sprite_bytes
        sprites_ram[sprite_name] = hex(sprite_ram)[2:].upper()

Function for actually drawing the sprite

In [2]:
from PIL import Image, ImageDraw, ImageColor, ImageFont
SPRITE_COLS = 24
SPRITE_ROWS = 21
CELL_WIDTH = 40
CELL_HEIGHT = 40


def paintSprite(sprite_data, colors):
    (background, background_text), (multicol0, multicol0_text), (multicol1,multicol1_text), (color, color_text) = colors
    colormap = {
        "00": background,
        "01": multicol0,
        "10": color,
        "11": multicol1,
    }
    text_colormap = {
        "00": background_text,
        "01": multicol0_text,
        "10": color_text,
        "11": multicol1_text,
    }
    
    image_width = CELL_WIDTH*SPRITE_COLS
    image_height = CELL_HEIGHT*SPRITE_ROWS
    img = Image.new( 'RGBA', (image_width+1, image_height+1))
    draw = ImageDraw.Draw(img)

    fnt = ImageFont.truetype("RobotoMono-Light.ttf", 40)
    bit_array = sprite_data
    # Remember that each bitpair in the bit_array is duplicated.
    # e.g. 01 appears as 01,01 so that we can treat each element as
    # a single bit.
    for y, l in enumerate(bit_array):
        for x,bit in enumerate(l):
            pixel_color = ImageColor.getrgb(c64_to_rgb[colormap[bit]])
            X = x * CELL_WIDTH
            Y = y * CELL_HEIGHT
            draw.rectangle((X, Y, X+CELL_WIDTH, Y+CELL_HEIGHT), 
                           fill=pixel_color, outline="black")
            b = bit[(x%2)] # Get the correct side of the bitpair
            draw.text((X+10, Y-8), b, font=fnt, fill=text_colormap[bit])
    return img


In [3]:
byte_literals = """.BYTE
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
"""

def generateSpriteDiagram(sprite_name, entry_num, colors):
    sprite_data = sprites_data[sprite_name]
    sprite_bytes = sprites_bytes[sprite_name]
    sprite_ram = sprites_ram[sprite_name]
    sprite_offset = sprites_offset[sprite_name]
    
    sprite_img = paintSprite(sprite_data, colors)

    img = Image.new('RGBA', (1530,900))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"URIDIUM/MEANIE/{('0'+str(entry_num))[-2:]}"
    label_fnt_size = 35
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt.rotate(90,  expand=1)
    img.paste(label, (35,254))

    # Sprite spec name
    label_text = f"SPEC:{sprite_name.ljust(14)}   OFFSET:${sprite_offset.upper()}  RAM:${sprite_ram}"
    label_fnt_size = 30
    label_fnt = ImageFont.truetype("DepartureMono-Regular.otf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size+20))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt.rotate(90,  expand=1)
    img.paste(label, (80,-440))


    # Sprite orientation
    azimuth, roll, elevation = ("000","000","000")
    label_text = f"AZIMUTH:{azimuth}°    ELEVATION:{elevation}°    ROLL:{roll}°"
    label_fnt_size = 30
    label_fnt = ImageFont.truetype("DepartureMono-Regular.otf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size+20))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt.rotate(90,  expand=1)
    img.paste(label, (120,-440))

    # Color Label
    label_text = "BASE_COLOR: "
    label_fnt_size = 30
    label_fnt = ImageFont.truetype("DepartureMono-Regular.otf", label_fnt_size)
    txt_width = len(label_text) * (label_fnt_size - 10)
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    pixel_color = ImageColor.getrgb(c64_to_rgb[colors[2][0]])
    draw.rectangle([(210,6), (229,29)], fill = pixel_color, outline="black")
    label = txt
    img.paste(label, (160,860))

    # MultiCol0 Label
    label_text = "MULTI_COLOR0: "
    label_fnt_size = 30
    label_fnt = ImageFont.truetype("DepartureMono-Regular.otf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    pixel_color = ImageColor.getrgb(c64_to_rgb[colors[1][0]])
    draw.rectangle([(250,6), (269,29)], fill = pixel_color, outline="black")
    label = txt
    img.paste(label, (490,860))

    # MultiCol1 Label
    label_text = "MULTI_COLOR1: "
    label_fnt_size = 30
    label_fnt = ImageFont.truetype("DepartureMono-Regular.otf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    pixel_color = ImageColor.getrgb(c64_to_rgb[colors[3][0]])
    draw.rectangle([(250,6), (269,29)], fill = pixel_color, outline="black")
    label = txt
    img.paste(label, (850,860))


    # Sprite byte literals
    label_text = byte_literals
    label_fnt_size = 35
    label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt_height =  len(sprite_bytes.split()) * (label_fnt_size+10)
    txt = Image.new('RGBA', (txt_width, txt_height))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="blue")
    label = txt
    img.paste(label, (1140,8))

    # Sprite bytes
    label_text = sprite_bytes
    label_fnt_size = 35
    label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt_height =  len(sprite_bytes.split()) * (label_fnt_size+10)
    txt = Image.new('RGBA', (txt_width, txt_height))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt
    img.paste(label, (1260,8))

    # Main Sprite image
    img.paste(sprite_img, (160,10))

    # Byte delimiter ticks
    draw = ImageDraw.Draw(img)
    vectors = [
        ((480,0), (480,10)), 
        ((800,0), (800,10)),
        #((450,850), (450,865)), 
        #((770,850), (770,865)),
    ]
    for start,end in vectors:
        draw.line((start,end), width=3, fill="black")
    return img

In [5]:
from colors_map import *
raw_level_color_schemes = """
        .BYTE M_GRAY1,M_GRAY3,M_ORANGE,M_YELLOW,M_ORANGE  ; Level 1
        .BYTE M_BLACK,M_GRAY1,M_LTBLUE,M_LTRED,M_RED      ; Level 2, Level 13
        .BYTE M_BROWN,M_ORANGE,M_ORANGE,M_LTGREEN,M_GREEN ; Level 3, Level 11
        .BYTE M_GRAY1,M_GRAY3,M_ORANGE,M_LTBLUE,M_BLUE    ; Level 4
        .BYTE M_GREEN,M_LTGREEN,M_ORANGE,M_LTBLUE,M_BLUE  ; Level 5
        .BYTE M_ORANGE,M_YELLOW,M_ORANGE,M_GRAY2,M_BLACK  ; Level 6
        .BYTE M_GRAY1,M_CYAN,M_LTGREEN,M_LTRED,M_RED      ; Level 7
        .BYTE M_BLACK,M_GRAY2,M_LTRED,M_LTBLUE,M_BLUE     ; Level 8
        .BYTE M_BLUE,M_LTBLUE,M_ORANGE,M_LTGREEN,M_GREEN  ; Level 9 , Level 14
        .BYTE M_GRAY1,M_GRAY2,M_GRAY2,M_LTRED,M_RED       ; Level 10
        .BYTE M_BROWN,M_ORANGE,M_ORANGE,M_LTGREEN,M_GREEN ; Level 3, Level 11
        .BYTE M_BLUE,M_CYAN,M_ORANGE,M_GRAY2,M_GRAY1      ; Level 12
        .BYTE M_BLACK,M_GRAY1,M_LTBLUE,M_LTRED,M_RED      ; Level 2, Level 13
        .BYTE M_BLUE,M_LTBLUE,M_ORANGE,M_LTGREEN,M_GREEN  ; Level 9 , Level 14
        .BYTE M_RED,M_LTRED,M_ORANGE,M_YELLOW,M_ORANGE    ; Level 15
"""
raw_level_color_schemes = [l[14:57].split(',') for l in raw_level_color_schemes.split('\n')][1:-1]
raw_level_color_schemes = [[x.strip() for x in l] for l in raw_level_color_schemes]
level_colors = [None]
for l in raw_level_color_schemes:
    colors = (color_constants[l[3]], "c64_white",color_constants[l[4]])
    level_colors += [colors]
level_colors[1]

('c64_yellow', 'c64_white', 'c64_orange')

In [6]:
!mkdir -p meanie_bullet_diagrams_list
!mkdir -p meanie_bullet_diagrams


In [7]:
for level, colors in enumerate(level_colors):
    if not level:
        continue
    for entry_num, sprite_name in enumerate(sprites_data):
        if not sprite_name:
            continue
        ccolors = (
            # cell color, text color
            ("c64_gray3", "white"),
            (colors[0], "darkgray"),
            (colors[1], "gray"),
            (colors[2], "white"),
        )
        img = generateSpriteDiagram(sprite_name, entry_num, ccolors)
        img.save(f"meanie_bullet_diagrams/{level}_{sprite_name}.png")
        img.save(f"meanie_bullet_diagrams_list/{level}_MEANIE_{entry_num}.png")

### Write out sprite images

In [10]:
from PIL import Image, ImageColor
SPRITE_COLS = 24
SPRITE_ROWS = 21
CELL_WIDTH = 40
CELL_HEIGHT = 40

def paintRawSprite(sprite, colors, verticalExpand=False):
    multicol0, multicol1,color = colors
    
    verticalExpansion = 2 if verticalExpand else 1
    SPRITE_COLS = 24
    SPRITE_ROWS = 21 * verticalExpansion

    colormap = {
        "01": multicol0,
        "10": color,
        "11": multicol1,
    }
    
    if sprite not in sprites_data:
        print(sprite)
        return
    
    image_width = SPRITE_COLS
    image_height = SPRITE_ROWS
    img = Image.new( 'RGBA', (image_width+1, image_height+1))
    pixels = img.load()

    bit_array = sprites_data[sprite]
    if verticalExpand:
        expanded_bit_array = []
        for a in bit_array:
            expanded_bit_array += [a,a]
        bit_array = expanded_bit_array
    
    for y, l in enumerate(bit_array):
        for x,bit in enumerate(l):
            if bit == "00":
                continue
            pixel_color = ImageColor.getrgb(c64_to_rgb[colormap[bit]])
            pixels[x,y] = pixel_color
    return img


In [9]:
!mkdir -p meanie_bullet_sprites

In [11]:
for i, colors in enumerate(level_colors):
    if not i:
        continue
    sprite_images = []
    for sprite_name in sprites_data:
        img = paintRawSprite(sprite_name, colors)
        sprite_images += [img]
        if sprite_name:
            img.save(f"meanie_bullet_sprites/{i}_{sprite_name}.png")


# Scratchpad